# Notebook 50 — Learning the Functional Map: `b ≈ Model[Γ_eff(x)]`

This notebook extends Notebook 49.

Notebook 49 showed that the stretching exponent `b` is better understood as a
**functional of the effective-rate process** `Γ_eff(x)` rather than a function
of a single scalar like CV.

This notebook takes the next step:

> Can we **learn** the mapping from the full shape of `Γ_eff(x)` to the fitted
> stretching exponent `b`?

We compare several predictive models:

- scalar baseline: `b ≈ f(CV)`
- low-dimensional shape model: `b ≈ f(PCA modes of Γ_eff(x))`
- flexible regression model: `b ≈ ML[Γ_eff(x)]`

## Main goals

1. generate a broad synthetic family of `Γ_eff(x)` curves,
2. compute the associated decay laws `S(x)`,
3. fit stretched exponentials and extract target values `b`,
4. learn predictive mappings from `Γ_eff(x)` to `b`,
5. compare scalar, linear, and nonlinear predictors.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error, r2_score


## Grid

In [ ]:
x = np.linspace(1e-3, 1.0, 400)
dx = x[1] - x[0]
rng = np.random.default_rng(42)


## Helper functions

In [ ]:
def stretched(x, a, b):
    return np.exp(-a * np.power(x, b))

def fit_stretched(x, S):
    popt, _ = curve_fit(
        stretched,
        np.clip(x, 1e-12, None),
        np.clip(S, 1e-12, 1.0),
        p0=[1.0, 1.0],
        bounds=([0.0, 0.1], [100.0, 5.0]),
        maxfev=50000,
    )
    a, b = [float(v) for v in popt]
    pred = stretched(x, a, b)
    mse = float(np.mean((S - pred) ** 2))
    return a, b, pred, mse

def build_S_from_gamma(gamma_x, x):
    integral = np.cumsum(gamma_x) * (x[1] - x[0])
    return np.exp(-integral)

def gamma_features(gamma_x, x):
    gamma_x = np.asarray(gamma_x, dtype=float)
    d1 = np.gradient(gamma_x, x)
    d2 = np.gradient(d1, x)
    mean = float(np.mean(gamma_x))
    std = float(np.std(gamma_x))
    cv = float(std / mean) if mean > 0 else np.nan
    return {
        "mean": mean,
        "std": std,
        "cv": cv,
        "slope_abs": float(np.mean(np.abs(d1))),
        "curvature_abs": float(np.mean(np.abs(d2))),
        "dynamic_range": float(np.max(gamma_x) - np.min(gamma_x)),
    }


## Synthetic family generator for `Γ_eff(x)`

In [ ]:
def synth_gamma_family(x, rng):
    # Randomly combine several interpretable components
    base = rng.uniform(1.4, 2.6)
    slope = rng.uniform(-1.0, 1.0)
    power_amp = rng.uniform(-0.7, 0.7)
    power_p = rng.uniform(0.35, 1.2)
    quad = rng.uniform(-0.9, 0.9)
    wave_amp = rng.uniform(0.0, 0.35)
    wave_freq = rng.choice([1, 2, 3])
    phase = rng.uniform(0, 2*np.pi)

    gamma = (
        base
        + slope * x
        + power_amp * np.power(x, power_p)
        + quad * (x - 0.5) ** 2
        + wave_amp * np.sin(2 * np.pi * wave_freq * x + phase)
    )

    # Keep rates positive
    gamma = gamma - np.min(gamma) + rng.uniform(0.6, 1.2)

    # Optional normalization to focus on shape more than absolute scale
    gamma = gamma / np.mean(gamma) * rng.uniform(1.6, 2.2)
    return gamma


## Build training dataset

In [ ]:
n_samples = 120
rows = []

for i in range(n_samples):
    gamma_x = synth_gamma_family(x, rng)
    S = build_S_from_gamma(gamma_x, x)
    a_fit, b_fit, S_fit, fit_mse = fit_stretched(x, S)
    feats = gamma_features(gamma_x, x)
    rows.append({
        "gamma": gamma_x,
        "S": S,
        "a_fit": a_fit,
        "b_fit": b_fit,
        "fit_mse": fit_mse,
        **feats,
    })

gamma_matrix = np.vstack([row["gamma"] for row in rows])
b_target = np.array([row["b_fit"] for row in rows], dtype=float)
cv_vals = np.array([row["cv"] for row in rows], dtype=float)

print("Samples:", len(rows))
print("b range:", float(np.min(b_target)), "to", float(np.max(b_target)))
print("mean fit mse:", float(np.mean([row["fit_mse"] for row in rows])))


## Plot representative synthetic `Γ_eff(x)` processes

In [ ]:
plt.figure(figsize=(8.4, 5.2))
for idx in [0, 5, 12, 25, 37, 58, 79, 101]:
    plt.plot(x, rows[idx]["gamma"], alpha=0.9, label=f'sample {idx}, b={rows[idx]["b_fit"]:.2f}')
plt.xlabel("x")
plt.ylabel("Γ_eff(x)")
plt.title("Representative training family of effective-rate processes")
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


## Plot representative resulting decays

In [ ]:
plt.figure(figsize=(8.4, 5.2))
for idx in [0, 5, 12, 25, 37, 58, 79, 101]:
    plt.plot(x, rows[idx]["S"], alpha=0.9, label=f'sample {idx}, b={rows[idx]["b_fit"]:.2f}')
plt.xlabel("x")
plt.ylabel("S(x)")
plt.title("Decay curves generated from synthetic Γ_eff(x)")
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


## Baseline model: `b ≈ f(CV)`

In [ ]:
def cv_model(cv, alpha, beta):
    return 1.0 / (1.0 + alpha * np.power(cv, beta))

popt_cv, _ = curve_fit(
    cv_model,
    cv_vals,
    b_target,
    p0=[1.0, 1.0],
    bounds=([0.0, 0.0], [100.0, 10.0]),
    maxfev=50000,
)
alpha_cv, beta_cv = [float(v) for v in popt_cv]
b_pred_cv = cv_model(cv_vals, alpha_cv, beta_cv)

print("CV baseline:")
print("alpha =", alpha_cv)
print("beta  =", beta_cv)
print("train MSE =", mean_squared_error(b_target, b_pred_cv))
print("train R^2 =", r2_score(b_target, b_pred_cv))


## PCA representation of `Γ_eff(x)`

In [ ]:
pca = PCA(n_components=5)
gamma_pca = pca.fit_transform(gamma_matrix)

print("Explained variance ratios:", np.round(pca.explained_variance_ratio_, 4))
print("Cumulative explained variance:", np.round(np.cumsum(pca.explained_variance_ratio_), 4))


## Plot the first two PCA modes

In [ ]:
plt.figure(figsize=(7.8, 5.0))
plt.plot(x, pca.components_[0], label="PC1")
plt.plot(x, pca.components_[1], label="PC2")
plt.xlabel("x")
plt.ylabel("mode amplitude")
plt.title("Leading PCA modes of Γ_eff(x)")
plt.legend()
plt.tight_layout()
plt.show()


## Models to compare

In [ ]:
X_cv = cv_vals.reshape(-1, 1)
X_pca2 = gamma_pca[:, :2]
X_pca5 = gamma_pca[:, :5]
X_full = gamma_matrix.copy()

models = {
    "CV only": Ridge(alpha=1e-6),
    "PCA(2) linear": Ridge(alpha=1.0),
    "PCA(5) linear": Ridge(alpha=1.0),
    "Full Γ linear": Ridge(alpha=5.0),
    "Full Γ random forest": RandomForestRegressor(
        n_estimators=400,
        max_depth=8,
        random_state=42,
        min_samples_leaf=2,
    ),
}
feature_sets = {
    "CV only": X_cv,
    "PCA(2) linear": X_pca2,
    "PCA(5) linear": X_pca5,
    "Full Γ linear": X_full,
    "Full Γ random forest": X_full,
}


## Leave-one-out cross-validation

In [ ]:
loo = LeaveOneOut()
cv_predictions = {name: np.zeros_like(b_target) for name in models}

for train_idx, test_idx in loo.split(b_target):
    for name, model in models.items():
        X = feature_sets[name]
        X_train, X_test = X[train_idx], X[test_idx]
        y_train = b_target[train_idx]
        model.fit(X_train, y_train)
        cv_predictions[name][test_idx[0]] = float(model.predict(X_test)[0])

scores = {}
for name, pred in cv_predictions.items():
    scores[name] = {
        "mse": float(mean_squared_error(b_target, pred)),
        "r2": float(r2_score(b_target, pred)),
    }

scores


## Compare model quality

In [ ]:
names = list(scores.keys())
mses = [scores[name]["mse"] for name in names]
r2s = [scores[name]["r2"] for name in names]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

axes[0].bar(names, mses)
axes[0].set_title("LOO prediction MSE")
axes[0].set_ylabel("MSE")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(names, r2s)
axes[1].set_title("LOO prediction R²")
axes[1].set_ylabel("R²")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()


## True vs predicted `b`

In [ ]:
plt.figure(figsize=(8.0, 5.5))
for name in names:
    plt.scatter(b_target, cv_predictions[name], s=36, alpha=0.65, label=name)

lims = [
    min(np.min(b_target), min(np.min(v) for v in cv_predictions.values())) - 0.02,
    max(np.max(b_target), max(np.max(v) for v in cv_predictions.values())) + 0.02,
]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("true b")
plt.ylabel("predicted b")
plt.title("Predicting the stretching exponent from Γ_eff(x)")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Residual comparison: CV-only vs best learned model

In [ ]:
best_model_name = min(scores.keys(), key=lambda k: scores[k]["mse"])
best_pred = cv_predictions[best_model_name]
resid_cv = b_target - cv_predictions["CV only"]
resid_best = b_target - best_pred

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), sharey=True)

axes[0].axhline(0.0, linestyle="--")
axes[0].scatter(cv_vals, resid_cv, s=30)
axes[0].set_xlabel("CV")
axes[0].set_ylabel("residual (true b - pred)")
axes[0].set_title("CV-only residuals")

axes[1].axhline(0.0, linestyle="--")
axes[1].scatter(cv_vals, resid_best, s=30)
axes[1].set_xlabel("CV")
axes[1].set_title(f"{best_model_name} residuals")

plt.tight_layout()
plt.show()

print("Best model:", best_model_name)


## Visualizing the learned low-dimensional map

In [ ]:
# Refit a smooth PCA(2) model on full data for visualization
pca2_model = Ridge(alpha=1.0)
pca2_model.fit(X_pca2, b_target)

pc1 = X_pca2[:, 0]
pc2 = X_pca2[:, 1]
pc1_grid = np.linspace(np.min(pc1), np.max(pc1), 140)
pc2_grid = np.linspace(np.min(pc2), np.max(pc2), 140)
PC1, PC2 = np.meshgrid(pc1_grid, pc2_grid)
B_grid = pca2_model.predict(np.column_stack([PC1.ravel(), PC2.ravel()])).reshape(PC1.shape)

plt.figure(figsize=(7.6, 5.6))
im = plt.pcolormesh(PC1, PC2, B_grid, shading="auto")
plt.scatter(pc1, pc2, c=b_target, edgecolors="white", s=55)
plt.colorbar(im, label="predicted b")
plt.xlabel("PC1 of Γ_eff(x)")
plt.ylabel("PC2 of Γ_eff(x)")
plt.title("Learned low-dimensional universality map")
plt.tight_layout()
plt.show()


## Feature importance for the best nonlinear model

In [ ]:
if best_model_name == "Full Γ random forest":
    rf = models["Full Γ random forest"]
    rf.fit(X_full, b_target)
    importances = rf.feature_importances_

    plt.figure(figsize=(8.0, 5.0))
    plt.plot(x, importances)
    plt.xlabel("x")
    plt.ylabel("importance")
    plt.title("Where Γ_eff(x) matters most for predicting b")
    plt.tight_layout()
    plt.show()
else:
    print("Best model is not the random forest in this run.")


## Compact summary

In [ ]:
lines = []
lines.append("Learning the functional map b ≈ Model[Γ_eff(x)]")
lines.append("")
for name in names:
    lines.append(
        f'- {name}: LOO MSE = {scores[name]["mse"]:.6e}, '
        f'LOO R^2 = {scores[name]["r2"]:.6f}'
    )
lines.append("")
lines.append(f"Best model: {best_model_name}")
lines.append("")
lines.append("Interpretation:")
lines.append("- Scalar summaries like CV capture some variation in b, but not all.")
lines.append("- Low-dimensional PCA representations improve prediction.")
lines.append("- Flexible models on the full Γ_eff(x) process can perform best.")
lines.append("- This supports the claim that the stretching exponent is learnable from the full rate process.")
print("\n".join(lines))


## Final conclusion

This notebook converts the functional-universality idea into a predictive learning problem.

The main lesson is:

- `b` is not only a function of a scalar disorder summary,
- it can be predicted from the **shape of the effective-rate process**,
- and low-dimensional or nonlinear models can learn that mapping.

This is the natural continuation after Notebook 49 because it turns

`b = Functional[Γ_eff(x)]`

into

`b ≈ LearnedModel[Γ_eff(x)]`.
